# Simulated KL-mode time series (turbulence and pyramid aliasing)

Specula frozen-flow runs of the aliasing study: `ristretto_main.yml` atmosphere (10 layers, seeing 0.9", L0 25 m, zenith 30°), 8.2 m pupil, 2 kHz (`dt` = 0.5 ms), 8 s = 16000 frames per run, noiseless cameras, 6" pyramid FoV, 48x48 pyramid at 900 nm. At every step the first N = 1300 KL modes of the turbulence are measured on the electric field and **removed exactly** by the DM, so the pyramid sees only the uncorrected (out-of-band) turbulence: its reconstructed modes are the **aliasing**. Wind directions of all layers rotated together by `rot` degrees; `seed` = phase-screen seed.

| folder (under `/raid1/mmenessini/results/WFSW/`) | pyramid modulation | files | arrays (T = 16000, N = 1300) |
|---|---|---|---|
| `alias_psd_mimo/modes/` | rMod 0 | `m0_rot{0,45,90,135}_seed{2001,2101}.npz` | **`atmo`**: true turbulence KL coefficients; `modes_900`: aliasing (VIS pyramid) |
| `alias_psd_rmod0/modes/` | rMod 0 | `r0_rot{0,45,90,135}_seed{1001,1101,1201,1301}.npz`, `r0_N300_rot*.npz` (N = 300) | `modes_900`, `modes_1300`: aliasing at 900 / 1300 nm (no `atmo`) |
| `alias_psd_rmod1/modes/` | rMod 1 | `r1_rot*_seed*.npz`, `r1_N300_rot*.npz` | as above |
| `alias_psd/modes/` | VIS rMod 3, NIR rMod 1 | `base_seed1-4`, `seeing045/120`, `L0_10`, single layers `layer_V10/V20`, `n48_N300`, `n24_N300`, `swap` (original wind directions) | as above |

Units: nm (OPD). `atmo` is exact (modal projection of the phase). `modes_*` = reconstructor (`calibration/WFSW/rec/pyr<r>_48x48_wvl<wl>_<N>modes_rec.fits`) x slopes, **not** optical-gain compensated (divide by the og of the mode for POL units). Runners: `run_mimo.py` / `run_rmod0.py` next to each `modes/` folder; analysis: `mimo_bound.py`, `aoeb_order.py` (measured aliasing spectra).

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

ROOT = '/raid1/mmenessini/results/WFSW'
files = sorted(glob.glob(f'{ROOT}/alias_psd_mimo/modes/m0_*.npz'))
print('\n'.join(files))

d = np.load(files[0])                      # one run: rMod 0, wind rotation 0, seed 2001
x = d['atmo'].astype(float)                # (T, N) true turbulence KL coefficients [nm]
a = d['modes_900'].astype(float)           # (T, N) pyramid-measured aliasing [nm], not og-compensated
dt = float(d['dt'])
t = np.arange(len(x)) * dt
print(f'keys {d.files}; T = {len(x)} frames, dt = {dt * 1e3:g} ms, N = {x.shape[1]} modes, seeing {float(d["seeing"]):g}"')
print('rms [nm] of modes 1, 10, 100, 1000:  turbulence', x[:, [0, 9, 99, 999]].std(0).round(2), '  aliasing', a[:, [0, 9, 99, 999]].std(0).round(3))

In [ ]:
modes = [1, 10, 100, 1000]                 # KL mode numbers (from 1 = tip)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for j, c in zip(modes, plt.cm.viridis(np.linspace(0, 0.9, len(modes)))):
    axs[0].plot(t, x[:, j - 1], color=c, lw=0.8, label=f'mode {j}')
    f, P = welch(x[:, j - 1], fs=1 / dt, nperseg=4000)
    axs[2].loglog(f[1:], P[1:], color=c, label=f'turbulence, mode {j}')
    f, P = welch(a[:, j - 1], fs=1 / dt, nperseg=4000)
    axs[2].loglog(f[1:], P[1:], color=c, ls='--', label=f'aliasing, mode {j}')
axs[0].set(xlabel='time [s]', ylabel='KL coefficient [nm]', title='True turbulence (atmo)')
axs[0].legend(fontsize=8)
jj = np.arange(1, x.shape[1] + 1)
axs[1].loglog(jj, x.std(0), label='turbulence (atmo)')
axs[1].loglog(jj, a.std(0), label='aliasing (modes_900)')
axs[1].set(xlabel='KL mode', ylabel='rms [nm]', title='Rms per mode')
axs[1].legend()
axs[2].set(xlabel='frequency [Hz]', ylabel='PSD [nm$^2$/Hz]', title='Temporal PSDs (solid: turbulence, dashed: aliasing)')
axs[2].legend(fontsize=7, ncol=2)
for ax in axs:
    ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()